# 03 — Göz Testi (F3 ADIM 4) 🚦 ÇIKIŞ KAPISI

20 sorgu (14 kanonik şablon + 6 serbest çizim) için **ilk 12 sonuç** 3×4 grid
olarak render edilir. Her karo: sembol, tarih, normalize skor, volatilite
etiketi ve vurgulanmış eşleşme penceresi.

Kontrol edilecekler:

* **K-09**: 12 karo 12 FARKLI olay mı, yoksa aynı grafiğin kaydırılmış kopyaları mı?
* **Sembol çeşitliliği**: `max 3/sembol` tutuyor mu?
* **Likidite**: ölü pariteler listeyi kirletiyor mu?
* **Uzunluk dengesi**: sonuçlar tek bir pencere uzunluğunda mı kümeleniyor?
  Kümeleniyorsa K-14 yol-uzunluğu normalizasyonu eksik/yanlış.

> **"Evet, bu çalışıyor" kararını bu defter VERMEZ.** Görseller üretilir,
> gözlemler yazılır; karar insana aittir (K-18).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

from engine.search.engine import SearchEngine
from engine.templates import load_all
from tools.f3_measure import render_top12, diversity_report, freehand_vector, FREEHAND_QUERIES

motor = SearchEngine()
sorgular = [(t.id, t.vector) for t in load_all().values()]
sorgular += [(f"serbest_{ad}", freehand_vector(ad)) for ad in FREEHAND_QUERIES]
print(f"{len(sorgular)} sorgu: {len(load_all())} şablon + {len(FREEHAND_QUERIES)} serbest çizim")

20 sorgu: 14 şablon + 6 serbest çizim


In [2]:
gozlem = []
for ad, q in sorgular:
    yol, sonuc = render_top12(motor, q, ad, TF, OUT, tolerance="medium")
    rapor = diversity_report(sonuc)
    rapor["sorgu"] = ad
    rapor["toplam"] = sonuc.total
    rapor["sure_ms"] = round(sonuc.elapsed_ms, 1)
    gozlem.append(rapor)
gozlem = pd.DataFrame(gozlem).set_index("sorgu")
gozlem[["toplam", "sonuc", "farkli_sembol", "max_per_symbol", "ortusen_cift", "genlik_medyan", "sure_ms"]]

,toplam,sonuc,farkli_sembol,max_per_symbol,ortusen_cift,genlik_medyan,sure_ms
sorgu,,,,,,,
ascending_triangle,0,0,0,0,0,NaN,85.6
bear_flag,249,12,12,1,0,73.342178,51.7
bull_flag,257,12,12,1,0,329.607803,54.9
descending_triangle,0,0,0,0,0,NaN,49.8
falling_wedge,485,12,12,1,0,62.500717,51.1
pennant,165,12,12,1,0,240.985939,52.2
rising_wedge,62,12,12,1,0,100.659512,52.3
symmetrical_triangle,0,0,0,0,0,NaN,53.6
cup_and_handle,87,12,12,1,0,56.998442,52.8


In [3]:
# K-09 doğrulaması: örtüşen çift SIFIR olmalı (post-NMS çalışıyorsa)
print("örtüşen çift toplamı :", int(gozlem.ortusen_cift.sum()), " (0 bekleniyor)")
print("max/sembol ihlali    :", int((gozlem.max_per_symbol > 3).sum()), " (0 bekleniyor)")
print("boş dönen sorgu      :", int((gozlem.sonuc == 0).sum()), f"/{len(gozlem)}")
print("ortalama farklı sembol:", round(gozlem.farkli_sembol.mean(), 1), "/ 12 karo")

örtüşen çift toplamı : 0  (0 bekleniyor)
max/sembol ihlali    : 0  (0 bekleniyor)
boş dönen sorgu      : 7 /20
ortalama farklı sembol: 7.2 / 12 karo


In [4]:
# Uzunluk dengesi — K-14 yol uzunluğu normalizasyonunun kanıtı
uzunluk = pd.DataFrame([g for g in gozlem.uzunluk_dagilimi]).fillna(0).astype(int)
uzunluk.index = gozlem.index
toplam_uzunluk = uzunluk.sum().sort_index()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([str(c) for c in toplam_uzunluk.index], toplam_uzunluk.values, color="#2b7a78")
ax.set_title("Sonuçların pencere uzunluğuna dağılımı (20 sorgu × 12 sonuç)")
ax.set_xlabel("pencere uzunluğu (bar)")
fig.tight_layout(); fig.savefig(OUT / "uzunluk_dengesi.png", dpi=110)
print(toplam_uzunluk.to_string())
kumelenme = toplam_uzunluk.max() > 0.7 * toplam_uzunluk.sum()
print("tek uzunlukta kümelenme var mı?", "EVET - K-14 normalizasyonu şüpheli" if kumelenme else "hayır")

24     97
40     31
64     12
100     5
tek uzunlukta kümelenme var mı? hayır


In [5]:
# Likidite: sonuçlarda ölü parite var mı?
tum_semboller = sorted({s for g in gozlem.index for s in []})
from collections import Counter
sayac = Counter()
for ad, q in sorgular:
    r = motor.search(q, timeframe=TF, tolerance="medium", limit=12)
    sayac.update(m.symbol for m in r.results)
print("sonuçlarda en sık geçen 15 sembol:")
for s, n in sayac.most_common(15):
    print(f"  {s:12s} {n:3d} kez")

sonuçlarda en sık geçen 15 sembol:
  WLDUSDT        3 kez
  ZECUSDT        3 kez
  AVAXUSDT       3 kez
  SCUSDT         3 kez
  ALGOUSDT       2 kez
  ICXUSDT        2 kez
  ALPHAUSDT      2 kez
  IOTAUSDT       2 kez
  XMRUSDT        2 kez
  MATICUSDT      2 kez
  PEPEUSDT       2 kez
  THETAUSDT      2 kez
  RUNEUSDT       2 kez
  ATOMUSDT       2 kez
  STORJUSDT      2 kez


### Gözlem notları

Grid PNG'leri: `notebooks/output/goz_testi_*.png`

Karar (insan): _______________